# DeiT-Ti from Scratch on CIFAR-100 (native 32×32)

> Trains **DeiT-Ti** from scratch on **CIFAR-100** at the dataset's native **32×32** resolution.
> No upsampling, no ImageNet pre-training.
>
> DeiT-Ti is the tiny variant from
> [*Training data-efficient image transformers & distillation through attention*](https://arxiv.org/abs/2012.12877)
> (Touvron et al., 2021): embed dim 192, depth 12, 3 heads, ~5M parameters.
>
> The paper's DeiT-Ti uses patch 16 on 224×224 images (196 tokens). At 32×32 that patch size
> would leave only **4 tokens**, which is not a working transformer. This notebook keeps the
> **Ti body** (dim / depth / heads) and uses **patch 4**, so you get `(32/4)^2 = 64` tokens —
> the usual CIFAR adaptation of ViT-Tiny. The architecture is otherwise identical.
>
> Default student is `timm.create_model('deit_tiny_patch16_224', pretrained=False, img_size=32,
> patch_size=4)`. A from-scratch implementation sits behind `config['model_source'] = 'scratch'`.

## What this notebook does

| Stage | Contents |
|---|---|
| Set-up | one `PLATFORM` flag for Kaggle / Colab / PSC / local |
| Data | `torchvision.datasets.CIFAR100` (auto-download, ~170 MB) |
| Model | DeiT-Ti body, patch 4, 32×32, 100 classes (~5.4M params) |
| Training | DeiT recipe adapted to CIFAR: RandomCrop+pad, RandAugment, MixUp/CutMix, EMA, AMP |
| Eval | Top-1 / Top-5, per-class accuracy, hflip TTA |

CIFAR-100 is 50k train / 10k test, 100 fine-grained classes, 32×32 RGB. Expect roughly
**60–75% top-1** for DeiT-Ti trained from scratch at 32px after 200–300 epochs. The paper's
90%+ CIFAR-100 number is ImageNet-pretrained DeiT-B fine-tuned at 224 — a different experiment.

# **SET-UP**

## Step 0: Choose your platform

CIFAR-100 is downloaded by `torchvision` — you do **not** need to add an Input dataset.

| `PLATFORM` | Data download goes to | Checkpoints & outputs |
|---|---|---|
| `'kaggle'` | `/kaggle/working/cifar100` | `/kaggle/working` |
| `'colab'`  | `/content/cifar100` | Google Drive (survives disconnects) |
| `'psc'`    | `$LOCAL/cifar100` | `~/deit_ti_cifar100` on `/jet/home` |
| `'local'`  | `./work/cifar100` | `./work` |
| `'auto'`   | detected | as above |

In [ ]:
# ============================ THE ONLY FLAG YOU NEED ============================
PLATFORM = 'auto'              # 'auto' | 'kaggle' | 'colab' | 'psc' | 'local'
# ===============================================================================

COLAB_MOUNT_DRIVE  = True
COLAB_DRIVE_SUBDIR = 'deit_ti_cifar100'
PSC_WORK_SUBDIR    = 'deit_ti_cifar100'

import os
import sys
import importlib.util
from pathlib import Path


def psc_scratch():
    return os.environ.get('LOCAL', '/local')


def detect_platform():
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.isdir('/kaggle/input'):
        return 'kaggle'
    if importlib.util.find_spec('google.colab') is not None:
        return 'colab'
    if os.path.isdir('/jet/home') or os.environ.get('LOCAL', '').startswith('/local'):
        return 'psc'
    return 'local'


PLATFORM = detect_platform() if PLATFORM == 'auto' else PLATFORM
assert PLATFORM in ('kaggle', 'colab', 'psc', 'local'), f"unknown PLATFORM: {PLATFORM}"
IS_KAGGLE = PLATFORM == 'kaggle'
IS_COLAB  = PLATFORM == 'colab'
IS_PSC    = PLATFORM == 'psc'
IS_LOCAL  = PLATFORM == 'local'


def resolve_work_dir():
    if IS_KAGGLE:
        return '/kaggle/working'
    if IS_COLAB:
        if COLAB_MOUNT_DRIVE:
            try:
                from google.colab import drive
                drive.mount('/content/drive')
                d = f"/content/drive/MyDrive/{COLAB_DRIVE_SUBDIR}"
                os.makedirs(d, exist_ok=True)
                return d
            except Exception as e:
                print(f"[warn] Drive mount failed ({e}); falling back to /content/work.")
        return '/content/work'
    if IS_PSC:
        return os.path.expanduser(f"~/{PSC_WORK_SUBDIR}")
    return './work'


def resolve_data_dir():
    if IS_KAGGLE:
        return '/kaggle/working/cifar100'
    if IS_COLAB:
        return '/content/cifar100'
    if IS_PSC:
        return f"{psc_scratch()}/cifar100"
    return os.path.join(resolve_work_dir(), 'cifar100')


ENV = {
    'platform' : PLATFORM,
    'work_dir' : resolve_work_dir(),
    'data_dir' : resolve_data_dir(),
}
os.makedirs(ENV['work_dir'], exist_ok=True)
os.makedirs(ENV['data_dir'], exist_ok=True)

print(f"Platform : {PLATFORM}")
print(f"Work dir : {ENV['work_dir']}")
print(f"Data dir : {ENV['data_dir']}")
if IS_PSC:
    scratch = psc_scratch()
    print(f"Scratch  : {scratch}")
    if not os.path.isdir(scratch) or not os.access(scratch, os.W_OK):
        print("[warn] $LOCAL is missing or not writable — you are probably on a login node.")

## Step 1: Verify the GPU

- **Kaggle**: *Settings → Accelerator → GPU T4* (one card is enough; DeiT-Ti at 32px is tiny).
- **Colab**: *Runtime → T4 GPU*.
- **PSC**: already on a GPU node (`interact -p GPU-shared --gres=gpu:v100-32:1 -t 8:00:00`).

In [ ]:
!nvidia-smi

## PSC / Bridges-2 setup (skip on Kaggle and Colab)

```bash
interact -p GPU-shared --gres=gpu:v100-32:1 -t 8:00:00 -A <your_allocation>
module load anaconda3
conda activate <your_env> && export PYTHONNOUSERSITE=1
cd /jet/home/$USER
jupyter notebook --no-browser --ip=0.0.0.0
```

## Step 2: Install / check packages

In [ ]:
import subprocess


def pip_install(packages):
    if not packages:
        return
    print("installing:", " ".join(packages))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=False)


def is_installed(pkg):
    try:
        return importlib.util.find_spec(pkg.replace('-', '_')) is not None
    except (ImportError, ValueError):
        return False


required = ['timm', 'wandb', 'torchinfo']
pip_install([p for p in required if not is_installed(p)])

import torch, torchvision
print("torch       :", torch.__version__)
print("torchvision :", torchvision.__version__)
import timm; print("timm        :", timm.__version__)
print("CUDA        :", torch.cuda.is_available(), "| devices:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print("   ", i, torch.cuda.get_device_name(i))

# Imports

In [ ]:
import os
import gc
import json
import math
import time
import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import torchvision
from torchvision.datasets import CIFAR100
from torchvision.utils import make_grid
import torchvision.transforms as T

import matplotlib.pyplot as plt

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", DEVICE)

# CIFAR-100 channel statistics (not ImageNet)
CIFAR_MEAN = (0.5071, 0.4867, 0.4408)
CIFAR_STD  = (0.2675, 0.2565, 0.2761)


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


torch.backends.cudnn.benchmark = True
try:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision('high')
except Exception:
    pass

# Config

DeiT-Ti body from the paper (Table 1): dim 192, 3 heads, 12 layers. Patch 4 on 32×32 is the
CIFAR substitution for the paper's patch 16 on 224×224 — same tokens-per-spatial-extent idea
(`image / patch = 8` in both cases is *not* the goal; 64 tokens is the usual CIFAR-Tiny setup).

`RandomResizedCrop(scale=0.08)` is an ImageNet transform and will zoom into a handful of
CIFAR pixels. Training uses `RandomCrop(32, padding=4)` instead, which is what DeiT's own
`datasets.py` does whenever `input_size <= 32`.

In [ ]:
config = {
    # ---- data -------------------------------------------------------------
    'data_dir'    : ENV['data_dir'],
    'num_classes' : 100,
    'image_size'  : 32,
    'num_workers' : min(os.cpu_count() or 4, 8),

    # ---- augmentation -----------------------------------------------------
    'auto_augment'       : 'rand-m9-mstd0.5-inc1',
    'train_interpolation': 'bicubic',
    'crop_padding'       : 4,          # CIFAR RandomCrop padding
    're_prob'            : 0.25,
    're_mode'            : 'pixel',
    're_count'           : 1,
    'repeated_aug'       : True,
    'ra_repeats'         : 3,

    # ---- mixup ------------------------------------------------------------
    'mixup_alpha'             : 0.8,
    'cutmix_alpha'            : 1.0,
    'mix_prob'                : 1.0,
    'mixup_cutmix_switch_prob': 0.5,
    'mixup_mode'              : 'batch',
    'label_smoothing'         : 0.1,

    # ---- model: DeiT-Ti body, CIFAR stem ---------------------------------
    'model_source'   : 'timm',      # 'timm' | 'scratch'
    'patch_size'     : 4,           # 32/4 = 8 grid -> 64 tokens
    'embed_dim'      : 192,
    'depth'          : 12,
    'num_heads'      : 3,
    'mlp_ratio'      : 4.0,
    'qkv_bias'       : True,
    'drop_rate'      : 0.0,
    'attn_drop_rate' : 0.0,
    'drop_path_rate' : 0.1,

    # ---- optimization ----------------------------------------------------
    'epochs'          : 300,
    'warmup_epochs'   : 10,
    'batch_size'      : 256,        # 32px + Tiny fits easily
    'grad_accum_steps': 1,
    'base_lr'         : 5e-4,
    'base_batch'      : 512,
    'warmup_lr'       : 1e-6,
    'min_lr'          : 1e-5,
    'weight_decay'    : 0.05,
    'grad_clip'       : None,
    'ema'             : True,
    'ema_decay'       : 0.9998,     # shorter horizon: more steps/epoch than IN-100 DeiT-S

    # ---- runtime ---------------------------------------------------------
    'amp'               : True,
    'use_dataparallel'  : False,
    'seed'              : 42,
    'max_runtime_hours' : 7.5 if IS_PSC else 8.5,
    'checkpoint_dir'    : os.path.join(ENV['work_dir'], 'checkpoints'),
    'resume_from'       : None,
    'smoke_test'        : False,

    # ---- logging ---------------------------------------------------------
    'use_wandb'     : False,
    'wandb_project' : "cifar100-deit-ti",
    'run_name'      : "deit_ti_scratch_cifar100_32",
}

config['eff_batch'] = config['batch_size'] * config['grad_accum_steps']
config['lr']        = config['base_lr'] * config['eff_batch'] / config['base_batch']

assert config['image_size'] % config['patch_size'] == 0
assert config['batch_size'] % 2 == 0, "timm Mixup requires an even batch size"
assert config['embed_dim'] % config['num_heads'] == 0

if config['smoke_test']:
    config.update(epochs=2, warmup_epochs=1, ema=False)

seed_everything(config['seed'])
os.makedirs(config['checkpoint_dir'], exist_ok=True)

n_tokens = (config['image_size'] // config['patch_size']) ** 2
print(json.dumps({k: str(v) for k, v in config.items()}, indent=2))
print(f"\nEffective batch: {config['eff_batch']}  ->  scaled LR: {config['lr']:.3e}")
print(f"Tokens: {n_tokens} patches + 1 cls  (grid {config['image_size'] // config['patch_size']}^2)")

# Data Augmentations

CIFAR pipeline, not the ImageNet one:

| Transform | Why |
|---|---|
| `RandomCrop(32, padding=4)` | the CIFAR default; DeiT uses this whenever `input_size <= 32` |
| `RandomHorizontalFlip` | standard |
| `RandAugment` `rand-m9-mstd0.5-inc1` | DeiT's policy |
| `Normalize(CIFAR-100 stats)` | dataset mean/std, not ImageNet |
| `RandomErasing(p=0.25, pixel)` | DeiT `--reprob` |
| MixUp / CutMix | batch-level, next section |

Validation is just tensor + normalize — the images are already 32×32.

In [ ]:
from timm.data import create_transform

# Official DeiT datasets.py: when input_size <= 32, swap RandomResizedCrop for
# RandomCrop(padding=4). Keep RandAugment + RandomErasing from the same recipe,
# but with CIFAR-100 mean/std so we are not using ImageNet statistics.


def create_transforms(cfg, is_training):
    if is_training:
        tfm = create_transform(
            input_size=cfg['image_size'],
            is_training=True,
            auto_augment=cfg['auto_augment'],
            interpolation=cfg['train_interpolation'],
            re_prob=cfg['re_prob'],
            re_mode=cfg['re_mode'],
            re_count=cfg['re_count'],
            mean=CIFAR_MEAN,
            std=CIFAR_STD,
        )
        tfm.transforms[0] = T.RandomCrop(cfg['image_size'], padding=cfg['crop_padding'])
        return tfm
    return T.Compose([
        T.ToTensor(),
        T.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])


train_transforms = create_transforms(config, is_training=True)
val_transforms   = create_transforms(config, is_training=False)
print("TRAIN:", train_transforms)
print("\nVAL  :", val_transforms)

# Datasets and DataLoaders

`CIFAR100` downloads ~170 MB on the first run into `ENV['data_dir']`. No Kaggle Input needed.

In [ ]:
class RASampler(torch.utils.data.Sampler):
    # Repeated-augmentation sampler, the num_replicas=1 case of deit/samplers.py.

    def __init__(self, dataset, num_repeats=3, shuffle=True, seed=0):
        if num_repeats < 1:
            raise ValueError("num_repeats should be greater than 0")
        self.num_samples = len(dataset)
        self.num_repeats = num_repeats
        self.shuffle = shuffle
        self.seed = seed
        self.epoch = 0
        self.num_selected_samples = (self.num_samples // 256) * 256 or self.num_samples

    def set_epoch(self, epoch):
        self.epoch = epoch

    def __iter__(self):
        if self.shuffle:
            g = torch.Generator()
            g.manual_seed(self.seed + self.epoch)
            indices = torch.randperm(self.num_samples, generator=g)
        else:
            indices = torch.arange(self.num_samples)
        indices = torch.repeat_interleave(indices, repeats=self.num_repeats, dim=0)
        return iter(indices[:self.num_selected_samples].tolist())

    def __len__(self):
        return self.num_selected_samples


train_dataset = CIFAR100(root=config['data_dir'], train=True,  download=True,
                         transform=train_transforms)
val_dataset   = CIFAR100(root=config['data_dir'], train=False, download=True,
                         transform=val_transforms)

classes = list(train_dataset.classes)
class_to_idx = dict(train_dataset.class_to_idx)
idx_to_name = {i: n for i, n in enumerate(classes)}
assert len(classes) == config['num_classes']

ra_sampler = RASampler(train_dataset, num_repeats=config['ra_repeats']) \
    if config['repeated_aug'] else None

train_loader = DataLoader(
    train_dataset,
    batch_size=config['batch_size'],
    sampler=ra_sampler,
    shuffle=(ra_sampler is None),
    num_workers=config['num_workers'],
    pin_memory=True,
    drop_last=True,
    persistent_workers=config['num_workers'] > 0,
    prefetch_factor=4 if config['num_workers'] > 0 else None,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=config['batch_size'] * 2,
    shuffle=False,
    num_workers=config['num_workers'],
    pin_memory=True,
    persistent_workers=config['num_workers'] > 0,
)

if ra_sampler is not None:
    unique = ra_sampler.num_selected_samples / config['ra_repeats']
    print(f"Repeated augmentation: {config['ra_repeats']}x -> "
          f"{ra_sampler.num_selected_samples:,} samples/epoch from ~{unique:,.0f} distinct images")

print(f"Train images : {len(train_dataset):,}  ({len(train_loader):,} batches)")
print(f"Val images   : {len(val_dataset):,}  ({len(val_loader):,} batches)")
print(f"Image shape  : {tuple(train_dataset[0][0].shape)}")
print(f"Classes      : {len(classes)}  e.g. {classes[:8]}")

## MixUp / CutMix

In [ ]:
from timm.data import Mixup

mixup_fn = Mixup(
    mixup_alpha=config['mixup_alpha'],
    cutmix_alpha=config['cutmix_alpha'],
    cutmix_minmax=None,
    prob=config['mix_prob'],
    switch_prob=config['mixup_cutmix_switch_prob'],
    mode=config['mixup_mode'],
    label_smoothing=config['label_smoothing'],
    num_classes=config['num_classes'],
)
mixup_active = config['mixup_alpha'] > 0 or config['cutmix_alpha'] > 0


def apply_batch_mix(images, labels):
    if not mixup_active:
        return images, labels
    return mixup_fn(images, labels)


print("Mixup active:", mixup_active)

# EDA and Viz

In [ ]:
print("Classes              :", len(classes))
print("Train / val          :", f"{len(train_dataset):,} / {len(val_dataset):,}")
print("Batch size           :", config['batch_size'], f"(effective {config['eff_batch']})")


def denormalize(x):
    mean = torch.tensor(CIFAR_MEAN).view(1, 3, 1, 1)
    std  = torch.tensor(CIFAR_STD).view(1, 3, 1, 1)
    return (x.cpu() * std + mean).clamp(0, 1)


def show_dataset_samples(loader, title, n=16, mix=False):
    images, labels = next(iter(loader))
    images, labels = images[:n], labels[:n]
    if mix:
        images, soft = apply_batch_mix(images, labels)
        labels = soft.argmax(1) if soft.ndim == 2 else soft
    grid = make_grid(denormalize(images), nrow=8, padding=2)
    fig, ax = plt.subplots(figsize=(10, 3.2))
    ax.imshow(grid.permute(1, 2, 0).numpy())
    ax.axis('off')
    ax.set_title(f"{title}\n" + " | ".join(str(idx_to_name[int(l)])[:10] for l in labels[:8]), fontsize=8)
    plt.tight_layout(); plt.show()


show_dataset_samples(train_loader, "Train (RandomCrop + RandAugment)")
show_dataset_samples(val_loader,   "Validation (no crop)")
show_dataset_samples(train_loader, "Train after MixUp / CutMix", mix=True)

# Model Architecture

DeiT-Ti is ViT-Tiny: dim 192, 3 heads (64 dim/head), 12 pre-norm blocks, MLP ratio 4.
The only CIFAR-specific change is the stem — a 4×4 stride-4 conv instead of 16×16 stride-16.

```
32×32 image
  → PatchEmbed 4×4  →  8×8 = 64 tokens × 192
  → + cls token + pos embed
  → 12 × (LN → Attention → LN → MLP) with stochastic depth
  → LN → linear 192→100
```

In [ ]:
def trunc_normal_(tensor, std=0.02):
    return nn.init.trunc_normal_(tensor, mean=0.0, std=std, a=-2 * std, b=2 * std)


class DropPath(nn.Module):
    def __init__(self, p: float = 0.0):
        super().__init__()
        self.p = p

    def forward(self, x):
        if self.p == 0.0 or not self.training:
            return x
        keep = 1.0 - self.p
        mask = x.new_empty((x.shape[0],) + (1,) * (x.ndim - 1)).bernoulli_(keep)
        return x * mask / keep


class PatchEmbed(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_chans=3, embed_dim=192):
        super().__init__()
        assert img_size % patch_size == 0
        self.img_size = img_size
        self.patch_size = patch_size
        self.grid_size = img_size // patch_size
        self.num_patches = self.grid_size ** 2
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        B, C, H, W = x.shape
        assert H == self.img_size and W == self.img_size, \
            f"Input {H}x{W} does not match model img_size {self.img_size}"
        x = self.proj(x)
        return x.flatten(2).transpose(1, 2)


class Attention(nn.Module):
    def __init__(self, dim, num_heads=3, qkv_bias=True, attn_drop=0.0, proj_drop=0.0):
        super().__init__()
        assert dim % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.attn_drop_p = attn_drop
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)
        self._fused = hasattr(F, 'scaled_dot_product_attention')

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        if self._fused:
            x = F.scaled_dot_product_attention(
                q, k, v, dropout_p=self.attn_drop_p if self.training else 0.0
            )
        else:
            attn = (q @ k.transpose(-2, -1)) * self.scale
            attn = self.attn_drop(attn.softmax(dim=-1))
            x = attn @ v
        x = x.transpose(1, 2).reshape(B, N, C)
        return self.proj_drop(self.proj(x))


class Mlp(nn.Module):
    def __init__(self, dim, hidden_dim, drop=0.0):
        super().__init__()
        self.fc1 = nn.Linear(dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, dim)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        return self.drop(self.fc2(self.drop(self.act(self.fc1(x)))))


class Block(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4.0, qkv_bias=True,
                 drop=0.0, attn_drop=0.0, drop_path=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, eps=1e-6)
        self.attn = Attention(dim, num_heads, qkv_bias, attn_drop, drop)
        self.norm2 = nn.LayerNorm(dim, eps=1e-6)
        self.mlp = Mlp(dim, int(dim * mlp_ratio), drop)
        self.drop_path = DropPath(drop_path)

    def forward(self, x):
        x = x + self.drop_path(self.attn(self.norm1(x)))
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x


class DeiT(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_chans=3, num_classes=100,
                 embed_dim=192, depth=12, num_heads=3, mlp_ratio=4.0, qkv_bias=True,
                 drop_rate=0.0, attn_drop_rate=0.0, drop_path_rate=0.1):
        super().__init__()
        self.num_classes = num_classes
        self.embed_dim = embed_dim
        self.patch_embed = PatchEmbed(img_size, patch_size, in_chans, embed_dim)
        num_patches = self.patch_embed.num_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.pos_drop = nn.Dropout(drop_rate)
        dpr = torch.linspace(0, drop_path_rate, depth).tolist()
        self.blocks = nn.ModuleList([
            Block(embed_dim, num_heads, mlp_ratio, qkv_bias, drop_rate, attn_drop_rate, dpr[i])
            for i in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim, eps=1e-6)
        self.head = nn.Linear(embed_dim, num_classes)
        self.apply(self._init_weights)
        trunc_normal_(self.pos_embed, std=0.02)
        trunc_normal_(self.cls_token, std=0.02)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)

    def no_weight_decay(self):
        return {'pos_embed', 'cls_token'}

    def forward_features(self, x):
        x = self.patch_embed(x)
        x = torch.cat([self.cls_token.expand(x.shape[0], -1, -1), x], dim=1)
        x = self.pos_drop(x + self.pos_embed)
        for blk in self.blocks:
            x = blk(x)
        return self.norm(x)

    def forward(self, x):
        x = self.forward_features(x)
        cls = x[:, 0]
        return {'feats': cls, 'out': self.head(cls)}


def deit_tiny(num_classes=100, img_size=32, **kw):
    defaults = dict(patch_size=4, embed_dim=192, depth=12, num_heads=3, mlp_ratio=4.0)
    defaults.update(kw)
    return DeiT(img_size=img_size, num_classes=num_classes, **defaults)


def fuse_logits(outputs):
    if isinstance(outputs, dict):
        return outputs['out']
    return outputs


def student_forward(model, images):
    out = model(images)
    return out['out'] if isinstance(out, dict) else out


def forward_logits(model, images):
    return fuse_logits(model(images))


def unwrap(model):
    return model.module if isinstance(model, (nn.DataParallel, nn.parallel.DistributedDataParallel)) else model

In [ ]:
if config['model_source'] == 'timm':
    import timm
    model = timm.create_model(
        'deit_tiny_patch16_224',
        pretrained=False,
        num_classes=config['num_classes'],
        img_size=config['image_size'],
        patch_size=config['patch_size'],
        drop_rate=config['drop_rate'],
        drop_path_rate=config['drop_path_rate'],
    )
else:
    model = deit_tiny(
        num_classes=config['num_classes'],
        img_size=config['image_size'],
        patch_size=config['patch_size'],
        embed_dim=config['embed_dim'],
        depth=config['depth'],
        num_heads=config['num_heads'],
        mlp_ratio=config['mlp_ratio'],
        qkv_bias=config['qkv_bias'],
        drop_rate=config['drop_rate'],
        attn_drop_rate=config['attn_drop_rate'],
        drop_path_rate=config['drop_path_rate'],
    )

model = model.to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Student : DeiT-Ti  ({config['model_source']})")
print(f"Params  : {n_params:,} ({n_params / 1e6:.2f} M)")

dummy = torch.randn(2, 3, config['image_size'], config['image_size'], device=DEVICE)
model.eval()
with torch.no_grad():
    logits = forward_logits(model, dummy)
print(f"Forward : {tuple(dummy.shape)} -> {tuple(logits.shape)}")
assert logits.shape == (2, config['num_classes'])

if config['use_dataparallel'] and DEVICE == 'cuda' and torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
    print(f"Wrapped in DataParallel over {torch.cuda.device_count()} GPUs")
else:
    print(f"Single-GPU training ({torch.cuda.device_count()} device(s) visible)")

gc.collect()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

# Loss | Optimizer | Scheduler

In [ ]:
from timm.loss import SoftTargetCrossEntropy, LabelSmoothingCrossEntropy
from timm.scheduler import create_scheduler_v2

if mixup_active:
    criterion = SoftTargetCrossEntropy()
else:
    criterion = LabelSmoothingCrossEntropy(smoothing=config['label_smoothing'])
val_criterion = nn.CrossEntropyLoss()

base_model = unwrap(model)
skip_keywords = set()
if hasattr(base_model, 'no_weight_decay'):
    skip_keywords = set(base_model.no_weight_decay())

decay, no_decay = [], []
for name, p in base_model.named_parameters():
    if not p.requires_grad:
        continue
    if p.ndim <= 1 or name.endswith('.bias') or any(k in name for k in skip_keywords):
        no_decay.append(p)
    else:
        decay.append(p)

optimizer = torch.optim.AdamW(
    [
        {'params': decay,    'weight_decay': config['weight_decay']},
        {'params': no_decay, 'weight_decay': 0.0},
    ],
    lr=config['lr'],
    betas=(0.9, 0.999),
    eps=1e-8,
)

scheduler, _ = create_scheduler_v2(
    optimizer,
    sched='cosine',
    num_epochs=config['epochs'],
    min_lr=config['min_lr'],
    warmup_lr=config['warmup_lr'],
    warmup_epochs=config['warmup_epochs'],
    cooldown_epochs=0,
)

amp_on = config['amp'] and DEVICE == 'cuda'
try:
    scaler = torch.amp.GradScaler('cuda', enabled=amp_on)
except (AttributeError, TypeError):
    scaler = torch.cuda.amp.GradScaler(enabled=amp_on)

print("train criterion:", type(criterion).__name__)
print(f"Decayed / no-decay tensors: {len(decay)} / {len(no_decay)}")
print(f"Scheduler: {type(scheduler).__name__} | {config['epochs']} epochs, "
      f"{config['warmup_epochs']} warm-up")

In [ ]:
class ModelEma:
    def __init__(self, model, decay=0.9998, warmup=True):
        self.module = copy.deepcopy(unwrap(model)).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)
        self.decay = decay
        self.warmup = warmup
        self.updates = 0

    @torch.no_grad()
    def update(self, model):
        self.updates += 1
        d = self.decay
        if self.warmup:
            d = min(d, (1 + self.updates) / (10 + self.updates))
        msd = unwrap(model).state_dict()
        for k, v in self.module.state_dict().items():
            src = msd[k].detach()
            if v.dtype.is_floating_point:
                v.mul_(d).add_(src.to(v.device), alpha=1.0 - d)
            else:
                v.copy_(src.to(v.device))

    def state_dict(self):
        return self.module.state_dict()

    def load_state_dict(self, sd):
        self.module.load_state_dict(sd)


model_ema = ModelEma(model, decay=config['ema_decay']) if config['ema'] else None
print("EMA:", "enabled" if model_ema else "disabled")

# Metrics

In [ ]:
class AverageMeter:
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = self.avg = self.sum = 0.0
        self.count = 0

    def update(self, val, n: int = 1):
        self.val = float(val)
        self.sum += float(val) * n
        self.count += n
        self.avg = self.sum / max(self.count, 1)


def topk_accuracy(logits, targets, topk=(1, 5)):
    if targets.ndim == 2:
        targets = targets.argmax(dim=1)
    maxk = min(max(topk), logits.size(1))
    batch_size = targets.size(0)
    _, preds = logits.topk(maxk, dim=1, largest=True, sorted=True)
    correct = preds.t().eq(targets.view(1, -1))
    out = []
    for k in topk:
        k = min(k, maxk)
        out.append(correct[:k].reshape(-1).float().sum(0) * 100.0 / batch_size)
    return out


def format_time(seconds):
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    return f"{h:d}h{m:02d}m{s:02d}s"

# Train and Validation Functions

In [ ]:
def train_epoch(model, dataloader, optimizer, scaler, device, criterion,
                ema=None, accum_steps=1, grad_clip=None, epoch=0, limit_batches=None):
    model.train()
    loss_meter, top1_meter, top5_meter = AverageMeter(), AverageMeter(), AverageMeter()
    n_batches = len(dataloader) if limit_batches is None else min(limit_batches, len(dataloader))
    progress = tqdm(dataloader, total=n_batches, desc=f"Train {epoch + 1}",
                    dynamic_ncols=True, leave=False)
    optimizer.zero_grad(set_to_none=True)
    amp_enabled = scaler.is_enabled()

    for i, (images, labels) in enumerate(progress):
        if i >= n_batches:
            break
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        hard_labels = labels
        images, labels = apply_batch_mix(images, labels)

        with torch.amp.autocast('cuda', enabled=amp_enabled):
            logits = student_forward(model, images)
            loss = criterion(logits, labels)

        scaler.scale(loss / accum_steps).backward()
        is_step = ((i + 1) % accum_steps == 0) or (i + 1 == n_batches)
        if is_step:
            if grad_clip is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            if ema is not None:
                ema.update(model)

        batch_loss = loss.item()
        if math.isfinite(batch_loss):
            acc1, acc5 = topk_accuracy(logits.detach().float(), hard_labels, topk=(1, 5))
            bs = images.size(0)
            loss_meter.update(batch_loss, bs)
            top1_meter.update(acc1.item(), bs)
            top5_meter.update(acc5.item(), bs)
        progress.set_postfix(loss=f"{loss_meter.avg:.4f}", top1=f"{top1_meter.avg:.2f}%",
                             lr=f"{optimizer.param_groups[0]['lr']:.2e}")

    return top1_meter.avg, top5_meter.avg, loss_meter.avg


@torch.no_grad()
def validate(model, dataloader, device, criterion, amp=True, desc="Val", return_preds=False):
    model.eval()
    loss_meter, top1_meter, top5_meter = AverageMeter(), AverageMeter(), AverageMeter()
    all_preds, all_targets = [], []
    for images, labels in tqdm(dataloader, desc=desc, dynamic_ncols=True, leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=amp and device == 'cuda'):
            logits = forward_logits(model, images)
        logits = logits.float()
        loss = criterion(logits, labels)
        acc1, acc5 = topk_accuracy(logits, labels, topk=(1, 5))
        bs = images.size(0)
        loss_meter.update(loss.item(), bs)
        top1_meter.update(acc1.item(), bs)
        top5_meter.update(acc5.item(), bs)
        if return_preds:
            all_preds.append(logits.argmax(1).cpu())
            all_targets.append(labels.cpu())
    if return_preds:
        return (top1_meter.avg, top5_meter.avg, loss_meter.avg,
                torch.cat(all_preds).numpy(), torch.cat(all_targets).numpy())
    return top1_meter.avg, top5_meter.avg, loss_meter.avg

# WandB (optional)

Set `config['use_wandb'] = True` and paste your key. Checkpoints are mirrored with `wandb.save`.

In [ ]:
WANDB_KEY = ""   # TODO: paste your key from wandb.ai/settings (leave empty to skip)

run = None
if config['use_wandb'] and WANDB_KEY:
    import wandb
    wandb.login(key=WANDB_KEY)
    run = wandb.init(project=config['wandb_project'], name=config['run_name'], config=config)
    print("WandB run:", run.url)
else:
    print("WandB disabled")

# Checkpointing and Resuming

`last.pth` is written every epoch. On Colab / PSC / local it lives on a persistent disk, so
re-running the notebook resumes. On Kaggle, Save Version or `wandb.save` the file, then set
`config['resume_from']` in the next session.

In [ ]:
CKPT_DIR = config['checkpoint_dir']
os.makedirs(CKPT_DIR, exist_ok=True)


def save_checkpoint(path, model, optimizer, scheduler, scaler, ema, epoch, metrics, history):
    torch.save({
        'model_state_dict'    : unwrap(model).state_dict(),
        'ema_state_dict'      : ema.state_dict() if ema is not None else None,
        'optimizer_state_dict': optimizer.state_dict() if optimizer is not None else None,
        'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
        'scaler_state_dict'   : scaler.state_dict() if scaler is not None else None,
        'epoch'               : epoch,
        'metrics'             : metrics,
        'history'             : history,
        'config'              : config,
        'classes'             : classes,
    }, path)


def load_checkpoint(path, model, optimizer=None, scheduler=None, scaler=None, ema=None,
                    device='cuda', strict=True):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    unwrap(model).load_state_dict(ckpt['model_state_dict'], strict=strict)
    if ema is not None and ckpt.get('ema_state_dict') is not None:
        ema.load_state_dict(ckpt['ema_state_dict'])
    if optimizer is not None and ckpt.get('optimizer_state_dict'):
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    if scaler is not None and ckpt.get('scaler_state_dict'):
        scaler.load_state_dict(ckpt['scaler_state_dict'])
    print(f"Loaded {path} | epoch {ckpt.get('epoch', -1) + 1} | metrics: {ckpt.get('metrics')}")
    return ckpt


start_epoch = 0
best_top1 = 0.0
history = {'epoch': [], 'lr': [], 'train_loss': [], 'train_top1': [],
           'val_loss': [], 'val_top1': [], 'val_top5': [], 'ema_top1': []}

resume_path = config['resume_from'] or os.path.join(CKPT_DIR, 'last.pth')
if resume_path and os.path.exists(resume_path):
    ckpt = load_checkpoint(resume_path, model, optimizer, scheduler=None, scaler=scaler,
                           ema=model_ema, device=DEVICE)
    start_epoch = ckpt.get('epoch', -1) + 1
    best_top1 = max(ckpt.get('metrics', {}).get('best_top1', 0.0), 0.0)
    loaded_history = ckpt.get('history') or {}
    if set(loaded_history) >= set(history):
        history = {k: list(loaded_history[k]) for k in history}
    scheduler.step(start_epoch)
    print(f"Resuming at epoch {start_epoch + 1}/{config['epochs']} "
          f"(best top-1 so far: {best_top1:.2f}%), lr {optimizer.param_groups[0]['lr']:.3e}")
else:
    print("No checkpoint found - training from scratch.")

# Training

DeiT-Ti at 32px is small. On a T4 expect well under a minute per epoch at batch 256, so 300
epochs fit in one Kaggle session. The loop still checkpoints every epoch and stops at
`max_runtime_hours`.

In [ ]:
if 'run_start_time' not in globals():
    run_start_time = time.time()

limit_batches = 20 if config['smoke_test'] else None
stop_reason = "completed"

for epoch in range(start_epoch, config['epochs']):
    print(f"\n=== Epoch {epoch + 1}/{config['epochs']} ===")
    epoch_start = time.time()
    curr_lr = optimizer.param_groups[0]['lr']

    if ra_sampler is not None:
        ra_sampler.set_epoch(epoch)

    train_top1, train_top5, train_loss = train_epoch(
        model, train_loader, optimizer, scaler, DEVICE, criterion,
        ema=model_ema,
        accum_steps=config['grad_accum_steps'],
        grad_clip=config['grad_clip'],
        epoch=epoch,
        limit_batches=limit_batches,
    )
    print(f"Train | top1 {train_top1:6.2f}% | top5 {train_top5:6.2f}% | "
          f"loss {train_loss:.4f} | lr {curr_lr:.3e}")

    val_top1, val_top5, val_loss = validate(
        model, val_loader, DEVICE, val_criterion, amp=config['amp'], desc=f"Val {epoch + 1}"
    )
    print(f"Val   | top1 {val_top1:6.2f}% | top5 {val_top5:6.2f}% | loss {val_loss:.4f}")

    ema_top1 = float('nan')
    if model_ema is not None:
        ema_top1, ema_top5, ema_loss = validate(
            model_ema.module, val_loader, DEVICE, val_criterion, amp=config['amp'], desc="Val EMA"
        )
        print(f"EMA   | top1 {ema_top1:6.2f}% | top5 {ema_top5:6.2f}% | loss {ema_loss:.4f}")

    score = max(val_top1, ema_top1 if model_ema is not None and not math.isnan(ema_top1) else 0.0)
    for key, value in [('epoch', epoch + 1), ('lr', curr_lr),
                       ('train_loss', train_loss), ('train_top1', train_top1),
                       ('val_loss', val_loss), ('val_top1', val_top1),
                       ('val_top5', val_top5), ('ema_top1', ema_top1)]:
        history[key].append(value)

    metrics = {'train_loss': train_loss, 'train_top1': train_top1, 'val_loss': val_loss,
               'val_top1': val_top1, 'val_top5': val_top5, 'ema_top1': ema_top1,
               'lr': curr_lr, 'best_top1': max(best_top1, score)}

    if run is not None:
        run.log(metrics, step=epoch + 1)

    with open(os.path.join(CKPT_DIR, 'history.json'), 'w') as f:
        json.dump(history, f, indent=2)

    last_path = os.path.join(CKPT_DIR, 'last.pth')
    save_checkpoint(last_path, model, optimizer, scheduler,
                    scaler, model_ema, epoch, metrics, history)
    if run is not None:
        wandb.save(last_path)

    if score > best_top1:
        best_top1 = score
        best_path = os.path.join(CKPT_DIR, 'best.pth')
        save_checkpoint(best_path, model, optimizer, scheduler,
                        scaler, model_ema, epoch, metrics, history)
        if run is not None:
            wandb.save(best_path)
        print(f"  -> new best top-1: {best_top1:.2f}% (saved best.pth)")

    scheduler.step(epoch + 1)

    elapsed = time.time() - run_start_time
    print(f"  epoch time {format_time(time.time() - epoch_start)} | "
          f"session elapsed {format_time(elapsed)} | best {best_top1:.2f}%")

    if elapsed > config['max_runtime_hours'] * 3600:
        stop_reason = "time budget reached"
        print(f"\nStopping early: {stop_reason}. Resume from {last_path}")
        break

print(f"\nTraining finished ({stop_reason}). Best val top-1: {best_top1:.2f}%")
if run is not None:
    run.finish()

# Results

In [ ]:
hist = pd.DataFrame(history)

fig, axes = plt.subplots(1, 3, figsize=(16, 3.6))
axes[0].plot(hist['epoch'], hist['train_loss'], label='train')
axes[0].plot(hist['epoch'], hist['val_loss'], label='val')
axes[0].set_title('Loss'); axes[0].set_xlabel('epoch'); axes[0].legend(); axes[0].grid(alpha=.3)

axes[1].plot(hist['epoch'], hist['train_top1'], label='train top-1')
axes[1].plot(hist['epoch'], hist['val_top1'], label='val top-1')
if hist['ema_top1'].notna().any():
    axes[1].plot(hist['epoch'], hist['ema_top1'], label='EMA top-1', ls='--')
axes[1].plot(hist['epoch'], hist['val_top5'], label='val top-5', alpha=.6)
axes[1].set_title('Accuracy (%)'); axes[1].set_xlabel('epoch'); axes[1].legend(); axes[1].grid(alpha=.3)

axes[2].plot(hist['epoch'], hist['lr'])
axes[2].set_title('Learning rate'); axes[2].set_xlabel('epoch'); axes[2].grid(alpha=.3)
plt.tight_layout(); plt.show()
hist.tail(10)

# Final Evaluation

In [ ]:
best_path = os.path.join(CKPT_DIR, 'best.pth')
if os.path.exists(best_path):
    ckpt = load_checkpoint(best_path, model, device=DEVICE)
    m = ckpt.get('metrics', {})
    if ckpt.get('ema_state_dict') is not None and \
       (m.get('ema_top1') or 0) > (m.get('val_top1') or 0):
        unwrap(model).load_state_dict(ckpt['ema_state_dict'])
        print("Using EMA weights for evaluation.")
else:
    print("best.pth not found - evaluating the in-memory model.")

val_top1, val_top5, val_loss, preds, targets = validate(
    model, val_loader, DEVICE, val_criterion, amp=config['amp'],
    desc="Final val", return_preds=True,
)
print(f"\nFinal | top-1 {val_top1:.2f}% | top-5 {val_top5:.2f}% | loss {val_loss:.4f}")

In [ ]:
@torch.no_grad()
def validate_tta(model, dataloader, device, amp=True):
    model.eval()
    top1, top5 = AverageMeter(), AverageMeter()
    all_probs, all_targets = [], []
    for images, labels in tqdm(dataloader, desc="TTA", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with torch.amp.autocast('cuda', enabled=amp and device == 'cuda'):
            logits = forward_logits(model, images).float().softmax(1)
            logits = logits + forward_logits(model, torch.flip(images, dims=[3])).float().softmax(1)
        logits = logits / 2
        a1, a5 = topk_accuracy(logits, labels, topk=(1, 5))
        top1.update(a1.item(), images.size(0))
        top5.update(a5.item(), images.size(0))
        all_probs.append(logits.cpu())
        all_targets.append(labels.cpu())
    return top1.avg, top5.avg, torch.cat(all_probs), torch.cat(all_targets)


tta_top1, tta_top5, tta_probs, tta_targets = validate_tta(model, val_loader, DEVICE, config['amp'])
print(f"hflip TTA | top-1 {tta_top1:.2f}% | top-5 {tta_top5:.2f}%  "
      f"(vs {val_top1:.2f}% / {val_top5:.2f}% without TTA)")

In [ ]:
per_class = pd.DataFrame({'target': targets, 'pred': preds})
per_class['correct'] = (per_class.target == per_class.pred).astype(float)
acc_by_class = per_class.groupby('target')['correct'].mean().mul(100)
acc_by_class.index = [str(idx_to_name[i])[:28] for i in acc_by_class.index]

print("Easiest classes:\n", acc_by_class.sort_values(ascending=False).head(10).round(1), "\n")
print("Hardest classes:\n", acc_by_class.sort_values().head(10).round(1))

fig, ax = plt.subplots(figsize=(11, 3))
ax.bar(range(len(acc_by_class)), acc_by_class.sort_values().values)
ax.axhline(val_top1, color='r', ls='--', label=f'mean {val_top1:.1f}%')
ax.set_xlabel('class (sorted)'); ax.set_ylabel('top-1 (%)')
ax.set_title('Per-class CIFAR-100 accuracy'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
images, labels = next(iter(val_loader))
with torch.no_grad(), torch.amp.autocast('cuda', enabled=config['amp'] and DEVICE == 'cuda'):
    logits = forward_logits(model, images[:16].to(DEVICE)).float()
pred = logits.argmax(1).cpu()

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
vis = denormalize(images[:16])
for i, ax in enumerate(axes.flat):
    ax.imshow(vis[i].permute(1, 2, 0).numpy())
    ok = pred[i] == labels[i]
    ax.set_title(f"{str(idx_to_name[int(pred[i])])[:12]}\n({str(idx_to_name[int(labels[i])])[:12]})",
                 fontsize=7, color='green' if ok else 'red')
    ax.axis('off')
plt.suptitle("prediction (ground truth)", fontsize=9)
plt.tight_layout(); plt.show()

# Export Artifacts

In [ ]:
OUT_DIR = ENV['work_dir']

top5_idx = tta_probs.topk(5, dim=1).indices.numpy()
pred_df = pd.DataFrame({
    'target': targets,
    'target_name': [str(idx_to_name[int(t)]) for t in targets],
    'pred'  : preds,
    'pred_name': [str(idx_to_name[int(p)]) for p in preds],
    'correct': (targets == preds).astype(int),
    'top5'  : [' '.join(map(str, row)) for row in top5_idx],
})
pred_df.to_csv(f"{OUT_DIR}/predictions_val.csv", index=False)
hist.to_csv(f"{OUT_DIR}/history.csv", index=False)
with open(f"{OUT_DIR}/history.json", 'w') as f:
    json.dump(history, f, indent=2)

torch.save({
    'model_state_dict': unwrap(model).state_dict(),
    'config': config,
    'classes': classes,
    'val_top1': val_top1,
    'val_top5': val_top5,
    'tta_top1': tta_top1,
}, f"{OUT_DIR}/deit_ti_cifar100.pth")

print("predictions_val.csv rows:", len(pred_df))
print(pred_df.head())

In [ ]:
MODEL = unwrap(model)
n_params = sum(p.numel() for p in MODEL.parameters())
README = f"""
# DeiT-Ti from scratch on CIFAR-100 (native 32x32)

- **Model**: DeiT-Ti body (dim {config['embed_dim']}, depth {config['depth']},
  {config['num_heads']} heads) with patch {config['patch_size']} on {config['image_size']}px
  ({n_params / 1e6:.2f}M params, {(config['image_size'] // config['patch_size']) ** 2} tokens).
  No pre-trained weights, no upsampling.
- **Data**: CIFAR-100 ({len(train_dataset):,} train / {len(val_dataset):,} test).
- **Training**: AdamW (lr {config['lr']:.2e}, wd {config['weight_decay']}),
  cosine schedule with {config['warmup_epochs']}-epoch warm-up, {config['epochs']} epochs,
  batch {config['eff_batch']}, {'fp16 AMP' if config['amp'] else 'fp32'},
  EMA decay {config['ema_decay']}.
- **Augmentations**: RandomCrop({config['image_size']}, padding={config['crop_padding']}),
  flip, RandAugment, RandomErasing(p={config['re_prob']}),
  MixUp/CutMix, label smoothing {config['label_smoothing']},
  repeated aug {config['ra_repeats']}x {'on' if config['repeated_aug'] else 'off'}.
- **Results**: val top-1 {val_top1:.2f}%, top-5 {val_top5:.2f}%, hflip-TTA {tta_top1:.2f}%.
- **Run**: {ENV['platform']}, work_dir `{ENV['work_dir']}`.
"""

with open(f"{OUT_DIR}/README.md", 'w') as f:
    f.write(README)
print(README)

# Notes

- **Why patch 4, not 16.** `(32/16)^2 = 4` tokens cannot do useful self-attention. Patch 4
  gives 64 tokens and is the standard CIFAR stem for ViT-Tiny / DeiT-Ti. The transformer
  blocks are the paper's DeiT-Ti.
- **Why not upsample.** You asked for native 32×32. Upsampling to 224 and using stock
  `deit_tiny_patch16_224` is a different (usually stronger) recipe because the positional
  embedding and the inductive bias of 196 tokens were designed for that scale.
- **Paper CIFAR numbers are not this experiment.** DeiT reports CIFAR-100 after ImageNet
  pre-training and 224px fine-tuning.